# Explicabilidade via LLM (OpenAI) com Contexto XAI (SHAP + LIME)

Este notebook combina resultados numericos de SHAP e LIME com a API da OpenAI para gerar explicacoes interpretaveis do modelo de deteccao de intrusao.

In [ ]:
# Importação de todas as bibliotecas que serão utilizadas
import pandas as pd
import numpy as np
import json
import shap
import lime
from lime import lime_tabular
from openai import OpenAI
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
# ==========================
# CONFIGURAÇÃO DA OPENAI
# ==========================
client = OpenAI(api_key="coloca a chave da API aqui")
LLM_MODEL = "gpt-5" 

# Leitura e Pre-processamento dos Dados

In [ ]:
# Define o dataset que será utilizado
df = pd.read_csv("./Network_logs.csv")

In [ ]:
# Cria uma cópia do dataset original
networkData = df.copy()

# Descarta as features IP de Origem/Destino (alta cardinalidade) e Intrusion (evitar overfitting)
networkData.drop(['Source_IP', 'Destination_IP', 'Intrusion'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
# Codificação para as features com valores não numéricos
categorical_cols = ['Request_Type', 'Protocol', 'User_Agent', 'Status', 'Port']
for col in categorical_cols:
    networkData[col] = networkData[col].astype('category')

for col in categorical_cols:
    print(f"{col} categories: {networkData[col].cat.categories.tolist()}")

for col in categorical_cols:
    networkData[col] = networkData[col].cat.codes

In [ ]:
# Codificação da variável Alvo (y): BotAttack -> 0; Normal -> 1; PortScan -> 2
target_encoder = LabelEncoder()
networkData['Scan_Type_Label'] = target_encoder.fit_transform(networkData['Scan_Type'])

label_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Label Mapping:", label_mapping)

networkData.drop(['Scan_Type'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
# Normalização do Payload_Size
scaler = StandardScaler()
networkData['Payload_Size'] = scaler.fit_transform(networkData[['Payload_Size']])

# Define features (X) e alvo (y)
X = networkData.drop(['Scan_Type_Label'], axis=1)
y = networkData['Scan_Type_Label']

# Particiona: 70% treino, 30% teste (estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# Aplica SMOTE para equilibrar classes no treinamento
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)
y_train = pd.Series(y_train.values.ravel(), name='Scan_Type_Label')

print('SMOTE aplicado com sucesso.\n')
print('Nova distribuição:\n')
print(y_train.value_counts())

# Treinamento do Modelo

In [ ]:
# Treina Random Forest
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Acurácia: {acc:.4f}")
print(classification_report(y_test, y_pred))

# Explicabilidade SHAP

In [ ]:
feature_names = list(X.columns)
class_names = list(target_encoder.classes_)  # ['BotAttack', 'Normal', 'PortScan']

# Seleciona amostra de teste para SHAP (max 200)
np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=min(200, len(X_test)), replace=False)
X_sample = X_test.loc[sample_idx]

# Calcula valores SHAP via TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Visualiza SHAP summary
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)

In [ ]:
# ==========================
# SHAP: IMPORTANCIA GLOBAL POR CLASSE
# ==========================
shap_global = {}
for cls_idx, cls_name in enumerate(class_names):
    mean_abs = np.abs(shap_values[:, :, cls_idx]).mean(axis=0)
    shap_global[cls_name] = {feat: round(float(val), 6) for feat, val in zip(feature_names, mean_abs)}

shap_global_json = json.dumps(shap_global, indent=2, ensure_ascii=False)
print("SHAP - Importancia Global por Classe:")
print(shap_global_json)

In [ ]:
# ==========================
# SHAP: EXPLICACOES LOCAIS (5 INSTANCIAS)
# ==========================
n_local = 5
shap_local = []
for idx in range(n_local):
    sample_row = X_sample.iloc[idx]
    real_idx = X_sample.index[idx]
    entry = {
        "instance_features": {feat: round(float(sample_row[feat]), 4) for feat in feature_names},
        "true_label": class_names[int(y_test.loc[real_idx])],
        "predicted_label": class_names[int(model.predict(sample_row.to_frame().T)[0])],
        "shap_values_per_class": {}
    }
    for cls_idx, cls_name in enumerate(class_names):
        entry["shap_values_per_class"][cls_name] = {
            feat: round(float(shap_values[idx, f_idx, cls_idx]), 6)
            for f_idx, feat in enumerate(feature_names)
        }
    shap_local.append(entry)

shap_local_json = json.dumps(shap_local, indent=2, ensure_ascii=False)
print("SHAP - Explicacoes Locais (5 instancias):")
print(shap_local_json[:1000], "...")

# Explicabilidade LIME

In [ ]:
# ==========================
# LIME: EXPLICACOES LOCAIS (mesmas 5 instancias do SHAP)
# ==========================
explainer_l = lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification'
)

lime_explanations = []
for idx in range(n_local):
    sample_row = X_sample.iloc[idx]
    real_idx = X_sample.index[idx]

    exp = explainer_l.explain_instance(
        sample_row.values,
        model.predict_proba,
        num_features=6
    )

    lime_explanations.append({
        "instance_index": int(real_idx),
        "true_label": class_names[int(y_test.loc[real_idx])],
        "predicted_label": class_names[int(model.predict(sample_row.to_frame().T)[0])],
        "local_prediction_proba": {
            cls: round(float(p), 4)
            for cls, p in zip(class_names, model.predict_proba(sample_row.to_frame().T)[0])
        },
        "feature_contributions": [
            {"rule": rule, "weight": round(float(w), 6)} for rule, w in exp.as_list()
        ]
    })

lime_explanations_json = json.dumps(lime_explanations, indent=2, ensure_ascii=False)
print("LIME - Explicacoes Locais (5 instancias):")
print(lime_explanations_json[:1000], "...")

# Construcao dos Metadados e Prompt para o LLM

In [ ]:
# ==========================
# METADADOS DO MODELO E DATASET
# ==========================
column_description = {
    "Port": "Porta utilizada na comunicação",
    "Request_Type": "Tipo de requisição",
    "Protocol": "Protocolo da Camada de Transporte (TCP, UDP ou ICMP)",
    "Payload_Size": "Tamanho do pacote (informação útil)",
    "User_Agent": "Agente utilizado na comunicação",
    "Status": "Status da requisição (Success ou Failure)",
    "Scan_Type_Label": "Classificação: Normal (1), BotAttack (0) ou PortScan (2)"
}

category_encoding = {
    "Request_Type": {"DNS": 0, "FTP": 1, "HTTP": 2, "HTTPS": 3, "SMTP": 4, "SSH": 5, "Telnet": 6},
    "Protocol": {"ICMP": 0, "TCP": 1, "UDP": 2},
    "User_Agent": {"Mozilla/5.0": 0, "Nikto/2.1.6": 1, "Wget/1.20.3": 2, "curl/7.68.0": 3, "nmap/7.80": 4, "python-requests/2.25.1": 5},
    "Status": {"Failure": 0, "Success": 1},
    "Port": {21: 0, 22: 1, 23: 2, 25: 3, 53: 4, 80: 5, 135: 6, 443: 7, 4444: 8, 6667: 9, 8080: 10, 31337: 11},
    "Scan_Type_Label": {"BotAttack": 0, "Normal": 1, "PortScan": 2}
}

stats = networkData.describe(include="all").to_string()

model_info = {
    "model_type": "Random Forest",
    "task": "Detecção de intrusão a partir de log. Cada entrada é classificada como: BotAttack (0) ou Normal (1) ou PortScan (2)",
    "target_variable": "Scan_Type_Label",
    "features": list(X.columns)
}

# Amostras de treinamento e previsões
train_sample = X_train.sample(50, random_state=42)
train_sample["Scan_Type_Label"] = y_train.loc[train_sample.index]
train_sample_json = train_sample.to_json(orient="records")

pred_sample = pd.DataFrame({"real": y_test, "predicted": y_pred})
pred_sample_json = pred_sample.sample(50, random_state=42).to_json(orient="records")

In [ ]:
# ==========================
# PROMPT ENRIQUECIDO COM DADOS XAI (SHAP + LIME)
# ==========================
prompt = f"""Voce e especialista em Inteligencia Artificial Explicavel (XAI) e Seguranca Cibernetica.
Sua tarefa e analisar um modelo de aprendizado de maquina utilizando resultados de metodos
formais de explicabilidade (SHAP e LIME) junto com os dados do modelo.

=========================
INFORMACOES DO MODELO
=========================
{json.dumps(model_info, indent=2, ensure_ascii=False)}

=========================
DESCRICAO DAS COLUNAS
=========================
{json.dumps(column_description, indent=2, ensure_ascii=False)}

=========================
CODIFICACAO: CATEGORIA -> VALOR NUMERICO
=========================
{json.dumps(category_encoding, indent=2, ensure_ascii=False)}

=========================
ESTATISTICA DO DATASET
=========================
{stats}

=========================
AMOSTRA DOS DADOS DE TREINAMENTO (50 registros)
=========================
{train_sample_json}

=========================
CLASSIFICACAO REAL vs PREVISTA (50 registros)
=========================
{pred_sample_json}

=========================
SHAP: IMPORTANCIA GLOBAL POR CLASSE
=========================
Media dos valores absolutos SHAP por feature para cada classe (calculado sobre {len(X_sample)} amostras):
{shap_global_json}

=========================
SHAP: EXPLICACOES LOCAIS (5 INSTANCIAS)
=========================
Valores SHAP por feature para cada classe em instancias especificas:
{shap_local_json}

=========================
LIME: EXPLICACOES LOCAIS (5 INSTANCIAS)
=========================
Contribuicoes de features segundo LIME para as mesmas instancias:
{lime_explanations_json}

=========================
TAREFA
=========================
Com base em TODOS os dados acima, forneca uma analise detalhada:

1. **Importancia Global das Features (baseado no SHAP):** Interprete o ranking de importancia
   SHAP para cada classe. Quais features dominam cada tipo de classificacao?

2. **Comparacao SHAP vs sua analise:** Compare o que os valores SHAP indicam com padroes
   que voce identifica nos dados de treinamento e previsoes. Ha concordancia?

3. **Explicacao Local de Previsoes (SHAP + LIME):** Para cada uma das 5 instancias,
   explique a previsao usando tanto os valores SHAP quanto as regras LIME.
   Destaque concordancias e divergencias entre os dois metodos.

4. **Analise de Classificacoes Incorretas:** Identifique classificacoes incorretas
   na amostra e use os dados SHAP/LIME para explicar possiveis causas.

5. **Insights de Seguranca Cibernetica:** Com base nos padroes detectados por SHAP e LIME,
   quais sao os indicadores mais fortes de BotAttack e PortScan nos logs de rede?

6. **Confiabilidade do Modelo:** Avalie a confiabilidade do modelo com base na coerencia
   entre SHAP, LIME e os padroes nos dados.

7. **Sugestoes de Melhoria:** Sugira melhorias para o modelo ou dataset com base nas
   evidencias XAI.

Estruture sua resposta em secoes claras e numeradas.
"""

print(f"Tamanho do prompt: {len(prompt)} caracteres")

# Chamada da API OpenAI com Contexto XAI

In [ ]:
# ==========================
# CHAMADA DA API OPENAI
# ==========================
system_msg = (
    "Voce e um especialista em Machine Learning, Explainable AI e Seguranca Cibernetica. "
    "Analise os resultados de SHAP e LIME fornecidos junto com os dados do modelo para "
    "explicar o comportamento do modelo de deteccao de intrusao. Sua analise deve ser "
    "tecnica, estruturada e adequada para um relatorio cientifico."
)

print(f"Consultando modelo: {LLM_MODEL}...")

response = client.responses.create(
    model=LLM_MODEL,
    input=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ],
    max_output_tokens=10000
)

explanation = ""
for item in response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                explanation += content.text

print(f"Resposta recebida: {len(explanation)} caracteres")

# Resultado: Explicacao do Modelo com Contexto XAI

In [ ]:
# ==========================
# RESULTADO
# ==========================
print("\n===== EXPLICACAO DO MODELO COM CONTEXTO XAI (SHAP + LIME) =====\n")
print(explanation)

In [ ]:
# ==========================
# SALVAR RESULTADO (opcional)
# ==========================
with open("resultado_openai_llm_xai_shap_lime.json", "w", encoding="utf-8") as f:
    json.dump({"model": LLM_MODEL, "explanation": explanation}, f, ensure_ascii=False, indent=2)
print("Resultado salvo em resultado_openai_llm_xai_shap_lime.json")